# Affine Term-Structure Models: The Vasicek Model

A walkthrough of the **Vasicek short-rate model**, the canonical example of
an **affine term-structure model (ATM)**: a short-rate model in which
zero-coupon bond prices come out as $P(t,T) = \exp\!\big(A(\tau) - B(\tau)\,r_t\big)$
for deterministic functions $A, B$ of time-to-maturity $\tau = T - t$.

We derive why the affine form falls out of the model's assumptions, solve
for $A(\tau)$ and $B(\tau)$ in closed form, simulate the short rate to
validate the bond-pricing formula by Monte Carlo, and look at what shapes of
yield curve the model can and can't produce.

This notebook is self-contained — everything is implemented inline with
NumPy/SciPy, no external package required.

## 1. Short rates and bond pricing

In a short-rate model, the entire term structure is derived from the
dynamics of the instantaneous risk-free rate $r_t$. Under the risk-neutral
measure $\mathbb{Q}$, the price at time $t$ of a zero-coupon bond paying \$1
at maturity $T$ is the discounted expectation of that payoff:

$$
P(t,T) = \mathbb{E}^{\mathbb{Q}}_t\!\left[\exp\!\left(-\int_t^T r_s\, ds\right)\right]
$$

Equivalently, by the Feynman-Kac theorem, $P(t,T)$ solves the term-structure
PDE

$$
\frac{\partial P}{\partial t} + \mu(r,t)\frac{\partial P}{\partial r}
+ \tfrac12 \sigma(r,t)^2 \frac{\partial^2 P}{\partial r^2} - r P = 0,
\qquad P(T,T) = 1
$$

for whatever drift $\mu$ and diffusion $\sigma$ the short rate follows. Once
we have $P(t,T)$, everything else (yields, forward rates, coupon bonds) is a
transformation of it — so the whole problem reduces to solving this one PDE.

## 2. The Vasicek model

Vasicek (1977) models the short rate as an **Ornstein-Uhlenbeck process**:

$$
dr_t = a(b - r_t)\, dt + \sigma\, dW_t
$$

- $b$ — the long-run mean level the rate reverts to
- $a > 0$ — the speed of mean reversion (larger $a$ = faster pull back to $b$)
- $\sigma$ — the (constant) instantaneous volatility

The drift $a(b - r_t)$ pulls $r_t$ back toward $b$ whenever it wanders away,
with a force proportional to the distance — this is what keeps the short
rate from drifting off to $\pm\infty$ the way plain Brownian motion would.

Because the SDE is linear in $r_t$ with constant coefficients, it can be
solved explicitly. Conditional on $r_t$, the rate at a future time $t+s$ is
**normally distributed**:

$$
r_{t+s} \mid r_t \;\sim\; \mathcal N\!\Big(
  \underbrace{r_t e^{-as} + b(1 - e^{-as})}_{\text{mean}},\;\;
  \underbrace{\frac{\sigma^2}{2a}\big(1 - e^{-2as}\big)}_{\text{variance}}
\Big)
$$

The mean is an exponentially-weighted blend of today's rate and the
long-run mean $b$; the variance saturates at $\sigma^2/(2a)$ as $s \to
\infty$ — the process has a well-defined *stationary distribution*,
$\mathcal N(b, \sigma^2/2a)$, unlike a driftless random walk.

The price of this tractability: $r_t$ is Gaussian, so it can go **negative**
with positive probability. That's the model's best-known limitation, and
we'll come back to it.

## 3. Why Vasicek is affine

"Affine" means the bond price takes the exponential-affine form

$$
P(t,T) = \exp\!\big(A(\tau) - B(\tau)\, r_t\big), \qquad \tau = T - t
$$

for functions $A, B$ that don't depend on $r_t$ — i.e. $\log P$ is *affine*
(linear plus a constant) in the state variable $r_t$. This isn't automatic;
it's a consequence of the drift and squared-diffusion both being affine
functions of $r$ (Duffie & Kan, 1996). Vasicek has

$$
\mu(r) = a(b-r) \quad \text{(affine in } r\text{)}, \qquad
\sigma(r)^2 = \sigma^2 \quad \text{(affine in } r\text{, with zero slope)}
$$

so it qualifies. Substituting the ansatz into the PDE from Section 1
(writing derivatives in $\tau = T-t$, so $\partial_t = -\partial_\tau$) and
matching terms that are constant in $r$ against terms proportional to $r$
gives two ODEs:

$$
B'(\tau) = 1 - aB(\tau), \qquad B(0) = 0
$$
$$
A'(\tau) = \tfrac12\sigma^2 B(\tau)^2 - ab\,B(\tau), \qquad A(0) = 0
$$

$B$'s equation is a simple linear ODE (no $B^2$ or $r$-dependent diffusion
term to complicate it — that's the payoff of $\sigma$ being constant), and
once $B$ is known, $A$ is just a quadrature. This is the general mechanism
behind every affine term-structure model: **CIR** ($\sigma(r) = \sigma
\sqrt r$, so $\sigma^2$ is affine with nonzero slope) and **Hull-White**
(Vasicek with time-dependent $a, b$) both reduce to the same style of
Riccati system, just with an extra term or time-dependence.

## 4. Closed-form solution

Solving the two ODEs above:

$$
B(\tau) = \frac{1 - e^{-a\tau}}{a}
$$

$$
A(\tau) = \left(b - \frac{\sigma^2}{2a^2}\right)\big(B(\tau) - \tau\big)
- \frac{\sigma^2 B(\tau)^2}{4a}
$$

so the zero-coupon bond price and continuously-compounded yield are

$$
P(t,T) = \exp\!\big(A(\tau) - B(\tau) r_t\big),
\qquad
y(t,T) = -\frac{\ln P(t,T)}{\tau} = \frac{B(\tau)\,r_t - A(\tau)}{\tau}
$$

Two sanity checks worth keeping in mind as we build the code:

- As $\tau \to 0$: $B(\tau) \to \tau$ and $A(\tau) \to 0$, so $y(t,T) \to
  r_t$ — the yield on an instant-maturity bond is just today's short rate.
- As $\tau \to \infty$: $B(\tau) \to 1/a$ and the yield converges to
  $R_\infty = b - \sigma^2/(2a^2)$ — a **long rate below the long-run mean**
  $b$ by a convexity term. That gap is the price of the rate's volatility:
  Jensen's inequality on the concave $\exp(-\cdot)$ discount factor makes
  bond prices (and hence long yields) respond asymmetrically to rate
  volatility.

## 5. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams["figure.dpi"] = 110
np.set_printoptions(precision=4, suppress=True)

# Vasicek parameters used throughout this notebook
a = 0.6        # speed of mean reversion
b = 0.03       # long-run mean short rate
sigma = 0.02   # volatility
r0 = 0.01      # today's short rate

print(f"a={a}  b={b}  sigma={sigma}  r0={r0}")
print(f"Stationary distribution: N(mean={b}, var={sigma**2 / (2*a):.6f}, std={np.sqrt(sigma**2/(2*a)):.4f})")

## 6. Simulating the short rate

Because the transition density is known exactly (Section 2), we can
simulate $r_t$ **without discretization bias** by drawing directly from the
conditional normal at each step, rather than using an Euler-Maruyama
approximation. We implement both, and compare them against the known
stationary distribution.

In [ ]:
def simulate_vasicek_exact(r0, a, b, sigma, T, n_steps, n_paths, rng):
    # Exact simulation using the known Gaussian transition density (no discretization bias)
    dt = T / n_steps
    mean_decay = np.exp(-a * dt)
    var_step = sigma**2 / (2 * a) * (1 - np.exp(-2 * a * dt))
    std_step = np.sqrt(var_step)

    paths = np.empty((n_paths, n_steps + 1))
    paths[:, 0] = r0
    Z = rng.standard_normal((n_paths, n_steps))
    for i in range(n_steps):
        paths[:, i + 1] = paths[:, i] * mean_decay + b * (1 - mean_decay) + std_step * Z[:, i]
    return paths


def simulate_vasicek_euler(r0, a, b, sigma, T, n_steps, n_paths, rng):
    # Euler-Maruyama discretization, for comparison
    dt = T / n_steps
    paths = np.empty((n_paths, n_steps + 1))
    paths[:, 0] = r0
    Z = rng.standard_normal((n_paths, n_steps))
    for i in range(n_steps):
        paths[:, i + 1] = paths[:, i] + a * (b - paths[:, i]) * dt + sigma * np.sqrt(dt) * Z[:, i]
    return paths


rng = np.random.default_rng(0)
T_sim, n_steps, n_paths = 10.0, 500, 20_000
t_grid = np.linspace(0, T_sim, n_steps + 1)

paths_exact = simulate_vasicek_exact(r0, a, b, sigma, T_sim, n_steps, n_paths, rng)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
for i in range(30):
    ax.plot(t_grid, paths_exact[i], lw=0.7, alpha=0.6)
ax.axhline(b, color="black", linestyle="--", lw=1.5, label=f"Long-run mean b={b}")
ax.set_xlabel("t")
ax.set_ylabel("$r_t$")
ax.set_title("Sample short-rate paths")
ax.legend()

ax = axes[1]
terminal = paths_exact[:, -1]
ax.hist(terminal, bins=80, density=True, alpha=0.5, color="#1f77b4", label="Simulated $r_T$")
x = np.linspace(terminal.min(), terminal.max(), 300)
stat_mean, stat_std = b, np.sqrt(sigma**2 / (2 * a))
from scipy.stats import norm
ax.plot(x, norm.pdf(x, stat_mean, stat_std), color="black", lw=1.5, label="Stationary $\\mathcal{N}(b,\\sigma^2/2a)$")
ax.set_xlabel("$r_T$")
ax.set_ylabel("Density")
ax.set_title(f"Terminal distribution at T={T_sim} (near-stationary)")
ax.legend()

fig.tight_layout()
plt.show()

print(f"Simulated mean: {terminal.mean():.4f}   Theoretical: {stat_mean:.4f}")
print(f"Simulated std:  {terminal.std():.4f}   Theoretical: {stat_std:.4f}")

## 7. Closed-form bond pricing

Implement $A(\tau)$, $B(\tau)$ from Section 4 and use them to price
zero-coupon bonds and read off the yield curve implied by today's short
rate $r_0$.

In [ ]:
def vasicek_AB(tau, a, b, sigma):
    B = (1 - np.exp(-a * tau)) / a
    A = (b - sigma**2 / (2 * a**2)) * (B - tau) - sigma**2 * B**2 / (4 * a)
    return A, B


def vasicek_bond_price(r, tau, a, b, sigma):
    A, B = vasicek_AB(tau, a, b, sigma)
    return np.exp(A - B * r)


def vasicek_yield(r, tau, a, b, sigma):
    A, B = vasicek_AB(tau, a, b, sigma)
    return (B * r - A) / tau


maturities = np.linspace(0.01, 30, 200)
yields = vasicek_yield(r0, maturities, a, b, sigma)
R_inf = b - sigma**2 / (2 * a**2)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(maturities, yields * 100, color="#1f77b4", lw=1.8, label="Vasicek yield curve")
ax.axhline(r0 * 100, color="#2ca02c", linestyle=":", label=f"$r_0$={r0:.2%}")
ax.axhline(R_inf * 100, color="black", linestyle="--", label=f"$R_\\infty$={R_inf:.2%}")
ax.set_xlabel("Maturity $\\tau$ (years)")
ax.set_ylabel("Yield (%)")
ax.set_title("Zero-coupon yield curve implied by the Vasicek model")
ax.legend()
fig.tight_layout()
plt.show()

## 8. Validating against Monte Carlo

The closed form comes from solving a PDE; let's confirm it agrees with the
definition it's supposed to satisfy — $P(0,T) = \mathbb{E}^{\mathbb{Q}}[\exp(-\int_0^T
r_s\, ds)]$ — by simulating short-rate paths, integrating them with the
trapezoidal rule, and averaging the discount factor across paths.

In [ ]:
def mc_bond_price(r0, a, b, sigma, T, n_steps, n_paths, rng):
    paths = simulate_vasicek_exact(r0, a, b, sigma, T, n_steps, n_paths, rng)
    dt = T / n_steps
    integral = np.trapz(paths, dx=dt, axis=1)   # \int_0^T r_s ds per path
    discount = np.exp(-integral)
    price = discount.mean()
    std_err = discount.std(ddof=1) / np.sqrt(n_paths)
    return price, std_err


rng = np.random.default_rng(42)
test_maturities = [1, 2, 5, 10, 20]
rows = []
for T in test_maturities:
    closed_form = vasicek_bond_price(r0, T, a, b, sigma)
    mc_price, mc_se = mc_bond_price(r0, a, b, sigma, T, n_steps=int(T * 100), n_paths=100_000, rng=rng)
    rows.append({
        "T": T,
        "closed_form": closed_form,
        "monte_carlo": mc_price,
        "mc_std_err": mc_se,
        "abs_diff": abs(closed_form - mc_price),
    })

df = pd.DataFrame(rows).set_index("T")
display(df)

## 9. What curve shapes can Vasicek produce?

The yield curve's shape is governed by where today's rate $r_0$ sits
relative to the model's long-run levels. Roughly:

- $r_0$ below the long-run mean $b$ → curve slopes **upward** (rates are
  expected to rise back toward $b$).
- $r_0$ above $b$ → curve slopes **downward**.
- $r_0 \approx b$ → curve is close to **flat**, near $R_\infty$.

Because $B(\tau)$ is a monotonic function of $\tau$ (it doesn't overshoot or
oscillate), a one-factor Vasicek curve is always **monotonic** — it can be
upward-sloping, downward-sloping, or flat, but it cannot produce a humped
(rises then falls) curve. That's a real limitation for fitting observed
curves, which do sometimes hump; multi-factor extensions exist precisely to
fix this.

In [ ]:
scenarios = {
    "r0 << b (upward)": 0.005,
    "r0 = b (flat-ish)": b,
    "r0 >> b (downward)": 0.07,
}

fig, ax = plt.subplots(figsize=(8, 4.5))
for label, r_start in scenarios.items():
    y = vasicek_yield(r_start, maturities, a, b, sigma)
    ax.plot(maturities, y * 100, lw=1.8, label=f"{label}, $r_0$={r_start:.2%}")
ax.axhline(R_inf * 100, color="black", linestyle="--", lw=1, label=f"$R_\\infty$={R_inf:.2%}")
ax.set_xlabel("Maturity $\\tau$ (years)")
ax.set_ylabel("Yield (%)")
ax.set_title("Yield curve shape depends on $r_0$ relative to $b$")
ax.legend()
fig.tight_layout()
plt.show()

### Effect of mean-reversion speed and volatility

$a$ controls how quickly the curve pulls in toward $R_\infty$ as maturity
grows (fast mean reversion → short rates dominate, curve flattens out
quickly); $\sigma$ controls how far $R_\infty$ sits below $b$ (more
volatility → bigger convexity discount at the long end).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
for a_val in [0.1, 0.3, 0.6, 1.5]:
    y = vasicek_yield(0.005, maturities, a_val, b, sigma)
    ax.plot(maturities, y * 100, lw=1.8, label=f"a={a_val}")
ax.set_xlabel("Maturity $\\tau$ (years)")
ax.set_ylabel("Yield (%)")
ax.set_title("Effect of mean-reversion speed $a$")
ax.legend()

ax = axes[1]
for sigma_val in [0.0, 0.01, 0.03, 0.06]:
    y = vasicek_yield(b, maturities, a, b, sigma_val)
    ax.plot(maturities, y * 100, lw=1.8, label=f"$\\sigma$={sigma_val}")
ax.set_xlabel("Maturity $\\tau$ (years)")
ax.set_ylabel("Yield (%)")
ax.set_title("Effect of volatility $\\sigma$ (starting from $r_0=b$)")
ax.legend()

fig.tight_layout()
plt.show()

## 10. Limitations and where affine models go from here

- **Negative rates.** $r_t$ is Gaussian, so $\mathbb P(r_t < 0) > 0$ always.
  This was long considered a flaw; it became a feature (or at least a
  non-issue) once negative policy rates became a real-world observation in
  the 2010s.
- **Monotonic curves only.** As noted above, a single Gaussian factor can't
  produce a humped term structure.
- **Doesn't fit the initial curve exactly.** With constant $a, b, \sigma$,
  the model produces *some* curve shape but generally won't match today's
  observed market curve at every maturity simultaneously.

Both of the last two points are addressed within the same affine framework:

- **CIR (Cox-Ingersoll-Ross)**: $dr_t = a(b-r_t)dt + \sigma\sqrt{r_t}\,dW_t$.
  Now $\sigma(r)^2 = \sigma^2 r$ is affine with *nonzero* slope, so $r_t$
  stays non-negative (under the Feller condition $2ab \ge \sigma^2$) and the
  $B(\tau)$ ODE becomes a genuine Riccati equation — still solvable in
  closed form, just with square roots in place of plain exponentials.
- **Hull-White**: Vasicek with $a, b$ (or just $b$) made deterministic
  functions of time, chosen so the model reproduces today's observed curve
  exactly, while keeping the same affine machinery and closed-form bond
  prices.
- **Multi-factor affine models** (e.g. two independent Vasicek/CIR factors
  summed together) recover humped curves and richer volatility dynamics
  while staying inside the same $P = \exp(A - B\cdot r)$ template — just
  with $r$ and $B$ becoming vectors.

The point of working through Vasicek in detail is that this same
affine-ansatz-into-Feynman-Kac recipe is exactly how all of these richer
models are solved.